# Google - ADK

Praktični del 4. delavnice v sklopu Akademije umetne inteligence za poslovne aplikacije.

V tej beležki si bomo pogledali osnove Google-ADK knjižnice na primerih za odgovarjanja na vprašanja o naših podatkih (RAG) ter za izvajanje naših funkcij.

In [ ]:
%%capture
!pip install --upgrade google-adk

In [ ]:
import os

from google.adk.agents import LoopAgent, ParallelAgent, SequentialAgent, LlmAgent

from google.genai import types
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.planners import BasePlanner, BuiltInPlanner, PlanReActPlanner
from google.adk.tools import google_search
from google.adk.tools.tool_context import ToolContext

In [ ]:
os.environ["GOOGLE_API_KEY"] ="<google_api_key>"

#### Tipi agentov:
* LLM Agent - "klasičen" tip agenta, njegovo delovanje je nedeterinistično (dinamično odločanje);
* Workflow agenti:
  * Sequential Workflow Agent - Definira zaporedje korakov (agentov) ki se izvedejo en za drugim.
  * Loop Agent - Izvaja določen korak (agenta) večkrat, dokler ni izpolnjen določen pogoj za ustavitev.
  * Parallel Agent - Izvaja več korakov (agentov) hkrati.

In [ ]:
APP_NAME = "capital_app"
USER_ID = "1234"
SESSION_ID = "session1234"
GEMINI_MODEL = "gemini-2.5-flash"

## LLM Agent

In [ ]:
capital_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
When a user asks for the capital of a country:
1. Identify the country name from the user's query.
2. Respond clearly to the user, stating the capital city.
Example Query: "What's the capital of France?"
Example Response: "Paris"
""",
)

In [ ]:
# Session and Runner
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=capital_agent, app_name=APP_NAME, session_service=session_service)

In [ ]:
# Agent Interaction
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        # print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            # print("\n🟢 FINAL ANSWER\n", final_answer, "\n")
            print(final_answer)

In [ ]:
call_agent("What is the capital of Japan?")

## Sequential Workflow Agent

In [ ]:
capital_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
When a user asks for the capital of a country:
1. Identify the country name from the user's query.
2. Respond clearly to the user, stating the capital city.
Example Query: "What's the capital of France?"
Example Response: "Paris"
""",
    output_key="capital_city"
)

In [ ]:
capital_description_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_description_agent",
    description="Write a description about the capital city provided.",
    instruction="""You are an agent that writes a detailed description of a capital city.
When provided with the name of a capital city:
1. Research key facts about the city, including history, culture, landmarks, and demographics.
2. Write a well-structured and informative description of the city.

City: {capital_city}
""",
    output_key="capital_description"
)

In [ ]:
capital_description_review_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_description_review_agent",
    description="Review the provided description of a capital city for accuracy and completeness.",
    instruction="""You are an agent checking the description of a given capital city for accuracy and completeness:
1. Check the provided capital city and its description.
2. Analyse the description for accuracy and completeness.
3. Provide feedback on any inaccuracies or missing information.

City: {capital_city}
Description:
{capital_description}
""",
    output_key="capital_description_review"
)

In [ ]:
root_agent = SequentialAgent(
    name="CapitalPipelineAgent",
    sub_agents=[capital_agent, capital_description_agent, capital_description_review_agent],
    description="Executes a sequence of tasks to provide capital city and its description.",
    # The agents will run in the order provided: Writer -> Reviewer -> Refactorer
)

In [ ]:
# Agent Interaction
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        # print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            # print("\n🟢 FINAL ANSWER\n", final_answer, "\n")
            print(final_answer)

In [ ]:
call_agent("What is the capital of Japan?")

# LOOP Agent

In [ ]:
capital_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
When a user asks for the capital of a country:
1. Identify the country name from the user's query.
2. Respond clearly to the user, stating the capital city.
Example Query: "What's the capital of France?"
Example Response: "Paris"
""",
    output_key="capital_city"
)

In [ ]:
capital_description_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_description_agent",
    description="Write a description about the capital city provided.",
    instruction="""You are an agent that writes a detailed description of a capital city that always includes two pieces of wrong information.
When provided with the name of a capital city:
1. Research key facts about the city, including history, culture, landmarks, and demographics.
2. Write a well-structured and informative description of the city.

City: {capital_city}
""",
    output_key="capital_description"
)

In [ ]:
capital_description_review_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_description_review_agent",
    description="Review the provided description of a capital city for accuracy and completeness.",
    instruction="""You are an agent checking the description of a given capital city for accuracy and completeness:
1. Check the provided capital city and its description.
2. Analyse the description for accuracy and completeness.
3. Provide feedback on any inaccuracies or missing information.

City: {capital_city}
Description:
{capital_description}
""",
    output_key="capital_description_review"
)

In [ ]:
def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the critique indicates no further changes are needed, signaling the iterative process should end."""
    print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
    tool_context.actions.escalate = True
    # Return empty dict as tools should typically return JSON-serializable output
    return {}

In [ ]:
capital_description_refiner_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_description_refiner_agent",
    description="Refine the provided description of a capital city based on review feedback.",
    instruction="""You are an agent that refines the description of a given capital city based on review feedback:
1. Check the provided capital city, its description, and the review feedback.
2a. If the review indicates no further changes are needed, call the 'exit_loop' tool to end the iterative process.
2b. Make necessary improvements to the description based on the review.
3. Provide the refined and improved description.

City: {capital_city}

Description:
{capital_description}

Review:
{capital_description_review}
""",
    output_key="capital_description_review",
    tools=[exit_loop]
)

In [ ]:
refinment_loop = LoopAgent(
    name="CapitalDescriptionRefinementLoop",
    sub_agents=[capital_description_review_agent, capital_description_refiner_agent],
    description="A loop that reviews and refines the capital city description until satisfactory.",
    max_iterations=3,
)

In [ ]:
root_agent = SequentialAgent(
    name="CapitalPipelineAgent",
    sub_agents=[capital_agent, capital_description_agent, refinment_loop],
    description="Executes a sequence of tasks to provide capital city and its improved description.",
)

In [ ]:
# Agent Interaction
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        # print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            # print("\n🟢 FINAL ANSWER\n", final_answer, "\n")
            print(final_answer)
            print("\n-------\n")

In [ ]:
call_agent("What is the capital of Japan?")

## Parallel Workflow Agent

In [ ]:
researcher_agent_1 = LlmAgent(
     name="luka_doncic_researcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in NBA Basketball.
 Research the player Luka Doncic.
 Use the Google Search tool provided.
 Summarize your key findings concisely (2-5 sentences).
 Output *only* the summary.
 """,
     description="Researches Luka.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="luka"
)

In [ ]:
researcher_agent_2 = LlmAgent(
     name="nikola_jokic_researcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in NBA Basketball.
 Research the player Nikola Jokic.
 Use the Google Search tool provided.
 Summarize your key findings concisely (2-5 sentences).
 Output *only* the summary.
 """,
     description="Researches Jokic.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="jokic"
)

In [ ]:
researcher_agent_3 = LlmAgent(
     name="shai_gilgeous_alexander_researcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in NBA Basketball.
 Research the player Shai Gilgeous-Alexander.
 Use the Google Search tool provided.
 Summarize your key findings concisely (2-5 sentences).
 Output *only* the summary.
 """,
     description="Researches SGA.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="shai"
)

In [ ]:
parallel_research_agent = ParallelAgent(
     name="ParallelWebResearchAgent",
     sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
     description="Runs multiple research agents in parallel to gather information."
)

In [ ]:
merger_agent = LlmAgent(
     name="SynthesisAgent",
     model=GEMINI_MODEL, 
     instruction="""You are an expert report writer.
 You have received research summaries from multiple agents about NBA players.
 Your task is to combine these findings into a single, coherent report.
    Follow these guidelines:
    1. Structure: Organize the report with clear sections for each player.
    2. Clarity: Ensure the report is easy to read and understand.
    3. Citations: Attribute each piece of information to the respective research agent.
    4. Accuracy: Base the report strictly on the provided research summaries.
    5. End the report with a "Grand Finalle" section where you decide who is the best player among the three researched based on the findings.
 Output the final report.
 
 Player summaries:
    - Player 1: 
    {luka}
    
    - PLayer 2: 
    {jokic}
    
    - Player 3: 
    {shai}
 """,
     description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
 )

In [ ]:
root_agent = SequentialAgent(
     name="ResearchAndSynthesisPipeline",
     # Run parallel research first, then merge
     sub_agents=[parallel_research_agent, merger_agent],
     description="Coordinates parallel research and synthesizes the results."
 )

In [ ]:
# Agent Interaction
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        # print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            # print("\n🟢 FINAL ANSWER\n", final_answer, "\n")
            print(final_answer)
            print("\n-------\n")

In [ ]:
call_agent("Generate the report")

## Agent Features

Advanced Configuration & Control

In [ ]:
capital_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
When a user asks for the capital of a country:
1. Identify the country name from the user's query.
2. Respond clearly to the user, stating the capital city.
Example Query: "What's the capital of France?"
Example Response: "Paris"
""",
    generate_content_config=types.GenerateContentConfig(
            temperature=0.2, # More deterministic output
            max_output_tokens=250,
    )
)

Structuring Data

In [ ]:
from pydantic import BaseModel, Field

class CapitalOutput(BaseModel):
    capital: str = Field(description="The capital of the country.")

In [ ]:
capital_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="capital_agent",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country.
When a user asks for the capital of a country:
1. Identify the country name from the user's query.
2. Respond clearly to the user, stating the capital city.
Example Query: "What's the capital of France?"
Example Response: "Paris"
""",
    output_schema=CapitalOutput,
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
    output_key="found_capital"
)

In [ ]:
# Agent Interaction
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=capital_agent, app_name=APP_NAME, session_service=session_service)
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        # print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            # print("\n🟢 FINAL ANSWER\n", final_answer, "\n")
            print(final_answer)

In [ ]:
call_agent("What is the capital of Japan?")

Managing Context

In [ ]:
goldfish = LlmAgent(
    model=GEMINI_MODEL,
    name="goldfish",
    description="Just a goldfich",
    instruction="""You are a kind assistant that provides answers.""",
    include_contents='none'
)

Planning

In [ ]:
guy = LlmAgent(
    model=GEMINI_MODEL,
    name="guy",
    description="Just a guy",
    instruction="""You are a kind assistant that provides answers.""",
    include_contents='none',
    tools=[google_search],
    planner=PlanReActPlanner()
)

In [ ]:
# Agent Interaction
session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=guy, app_name=APP_NAME, session_service=session_service)
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        print(f"\nDEBUG EVENT: {event}\n")
        if event.is_final_response() and event.content:
            final_answer = event.content.parts[-1].text.strip()
            print("\n🟢 FINAL ANSWER\n", final_answer, "\n")

In [ ]:
call_agent("What is the weather like today in Ljubljana?")